In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment
import time
import re
import numpy as np
from io import StringIO

In [2]:
BASE_URL = "https://www.sports-reference.com"
START_URL = f"{BASE_URL}/cbb/seasons/men/2026-school-stats.html"

def get_team_links():
    print("Fetching school list...")
    # It's good practice to add a User-Agent so the site doesn't think you're a bot immediately
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(START_URL, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    
    table = soup.find("table", {"id": "basic_school_stats"})
    links = []
    
    for row in table.find("tbody").find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue
        school_cell = row.find("td", {"data-stat": "school_name"})
        if school_cell and school_cell.find("a"):
            links.append({
                "school": school_cell.text,
                "url": BASE_URL + school_cell.find("a")["href"]
            })
    return links

# Use a real User-Agent to avoid getting flagged
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

def scrape_advanced_table(url):
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # 1. Look for the table in the normal HTML first
    table = soup.find('table', id='advanced')
    
    # 2. If not found, search inside the HTML comments
    if not table:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if 'id="advanced"' in comment:
                # Create a mini-soup from the comment string
                comment_soup = BeautifulSoup(comment, 'html.parser')
                table = comment_soup.find('table', id='advanced')
                break
                
    if table:
        # read_html returns a list of dataframes; we want the first one
        df = pd.read_html(str(table))[7]
        
        # Clean up the double-header (MultiIndex)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(0)
            
        return df
    return None

# Example usage for Abilene Christian
acu_url = "https://www.sports-reference.com/cbb/schools/abilene-christian/men/2026.html"
df = scrape_advanced_table(START_URL)

if df is not None:
    print("Successfully captured Advanced Table!")
    print(df.head())
else:
    print("Still couldn't find the table.")

Still couldn't find the table.


In [4]:
BASE_URL = "https://www.sports-reference.com"
START_URL = f"{BASE_URL}/cbb/seasons/men/2026-school-stats.html"

# Use a real User-Agent to avoid getting flagged
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def get_team_links():
    print("Fetching school list...")
    response = requests.get(START_URL, headers=HEADERS)
    soup = BeautifulSoup(response.content, "html.parser")
    
    table = soup.find("table", {"id": "basic_school_stats"})
    links = []
    
    # Check if table exists before trying to iterate
    if not table:
        print("Could not find the basic_school_stats table.")
        return links

    for row in table.find("tbody").find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue
        school_cell = row.find("td", {"data-stat": "school_name"})
        if school_cell and school_cell.find("a"):
            links.append({
                "school": school_cell.text,
                "url": BASE_URL + school_cell.find("a")["href"]
            })
    return links


def scrape_advanced_table(url):
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # 1. Look for the table in the normal HTML first
    table = soup.find('table', id='advanced')
    
    # 2. If not found, search inside the HTML comments
    if not table:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if 'id="advanced"' in comment:
                # Create a mini-soup from the comment string
                comment_soup = BeautifulSoup(comment, 'html.parser')
                table = comment_soup.find('table', id='advanced')
                break
                
    if table:
        # Wrap string in StringIO to avoid pandas warnings, and select index [0]
        html_string = StringIO(str(table))
        df = pd.read_html(html_string)[0]
        
        # Clean up the double-header (MultiIndex)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(0)
            
        return df
        
    return None


# Example usage for Abilene Christian
acu_url = "https://www.sports-reference.com/cbb/schools/abilene-christian/men/2026.html"

# Fix: Pass acu_url instead of START_URL
df = scrape_advanced_table(acu_url)

if df is not None:
    print("Successfully captured Advanced Table!")
    print(df.head())
else:
    print("Still couldn't find the table.")

Still couldn't find the table.
